# GAN-RL Protein Function Prediction
**Goal:** Beat ProtHGT-ESM2 Biological Process Fmax (baseline: 0.7489)

**Workflow:** Cell 1 → Cell 11 in order. Checkpoints are saved to Drive after each phase.

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

In [ ]:
# ── Cell 2: Install Dependencies ───────────────────────────────────────────────
# Colab already has PyTorch — do NOT reinstall it.
# We detect the pre-installed torch/CUDA version and pull matching PyG wheels.
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]             # e.g. '2.3.0'
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') # e.g. 'cu121'
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
print(f'Detected: torch={torch_ver}, cuda={cuda_tag}')
print(f'PyG wheel URL: {pyg_url}')

print('Installing torch_geometric ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)

print('Installing PyG sparse extensions ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', pyg_url, '-q'
], check=True)

print('Installing other deps ...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'networkx', 'pyyaml', 'obonet',
    'scikit-learn', 'tqdm', 'pandas',
    'matplotlib', 'tensorboard', '-q'], check=True)

print('Done. No runtime restart needed.')

In [ ]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT     = '/content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/'
CHECKPOINT_DIR = '/content/drive/MyDrive/Poster code/checkpoints/'
LOG_CSV        = os.path.join(CHECKPOINT_DIR, 'training_log.csv')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Sanity check — verify ESM2 split files are in place
esm2_dir = os.path.join(DRIVE_ROOT, 'alternative_protein_embeddings/esm2/')
expected = [
    'prothgt-esm2-train-graph.pt',
    'prothgt-esm2-val-graph.pt',
    'prothgt-esm2-test-graph.pt',   # optional — val used as proxy if missing
]
for f in expected:
    path = os.path.join(esm2_dir, f)
    exists = os.path.exists(path)
    tag = 'OK' if exists else ('MISSING (optional)' if 'test' in f else 'MISSING')
    print(f'  {f}: {tag}')

print(f'\nDRIVE_ROOT:     {DRIVE_ROOT}')
print(f'CHECKPOINT_DIR: {CHECKPOINT_DIR}')

In [ ]:
# ── Cell 4: Clone Repository ───────────────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/Prot_B_poster'
REPO_URL = 'https://github.com/Drjay806/Prot_B_poster.git'

if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, 'src')):
    print('Repo already present — pulling latest ...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print('Cloning repo ...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR cloning repo:')
        print(result.stderr)
        raise RuntimeError('Git clone failed — push your code first.')
    print(result.stdout)

# Install src as an editable package — fixes "No module named src" permanently.
# pip install -e registers src/ in Python's site-packages so any cell can import it
# without sys.path hacks, even after Drive remounts or kernel restarts within the session.
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR, '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip install -e failed:', result.stderr)
else:
    print('src package installed.')

print(f'Repo ready at {REPO_DIR}')
print('Contents:', os.listdir(REPO_DIR))

In [ ]:
# ── Cell 5: Load Config ────────────────────────────────────────────────────────
import yaml, os

config_path = os.path.join(REPO_DIR, 'configs/default.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Override paths with what we set above
cfg['data']['drive_root'] = DRIVE_ROOT
cfg['data']['checkpoint_dir'] = CHECKPOINT_DIR

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Target ontology:', cfg['data']['ontology'])
print('Target node type:', cfg['data']['target_type'])

In [ ]:
# ── Cell 6: Load Data + Build Hierarchy Tables ────────────────────────────────
# Data is loaded onto CPU. CompGCN moves only the relevant ~150 MB of node
# features to GPU per forward pass, freeing ~3 GB of VRAM for activations.

from src.data.loader import load_prothgt_splits
from src.data.go_hierarchy import build_ancestor_table, build_propagation_edges

splits = load_prothgt_splits(
    drive_root=cfg['data']['drive_root'],
    ontology=cfg['data']['ontology'],
    device='cpu',   # keep on CPU — encoder handles GPU placement internally
)
train_data, val_data, test_data = splits.train, splits.val, splits.test
target_type = splits.target_type

print(f'\nNode types: {train_data.node_types}')
print(f'Proteins:   {train_data["Protein"].x.shape[0]:,}')
print(f'GO terms:   {train_data[target_type].x.shape[0]:,}')

print('\nBuilding GO ancestor table (used by RL reward) ...')
ancestor_table = build_ancestor_table(train_data, target_type=target_type, cache=True)

print('Building propagation edge list (used by evaluation) ...')
prop_edges = build_propagation_edges(train_data, target_type=target_type, cache=True)
print(f'Ready. {len(ancestor_table):,} GO terms, {len(prop_edges):,} hierarchy edges.')

In [ ]:
# Cell 6b: Verify Split Disjointness
# ProtHGT uses a transductive edge-level split: proteins are shared across splits
# (needed for GNN message passing), but the specific annotation edges are held out.
# We check that (protein, GO) pairs don't overlap between train and test.

# Clear .pyc cache so any freshly pulled source files are used
import subprocess, importlib
subprocess.run(["find", "/content/Prot_B_poster", "-name", "*.pyc", "-delete"], capture_output=True)

import src.data.graph_builder as _gb
importlib.reload(_gb)
from src.data.graph_builder import validate_split_disjointness

print("Checking split disjointness...")
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        validate_split_disjointness(train_data, test_data, ttype)
    except (KeyError, ValueError) as e:
        print(f"  {ttype}: {e}")
print("Split validation complete.")


In [ ]:
# Cell 6c: Pre-compute Information Content Vectors (required for Smin metric)
# IC[t] = -log2(freq_train(t) / N_proteins), computed after label propagation.
# One call per ontology; results stored in ic_vecs dict keyed by ontology shortname.
from src.evaluation.smin import compute_information_content
from src.data.go_hierarchy import build_propagation_edges

ic_vecs = {}
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        pe = build_propagation_edges(train_data, target_type=ttype, cache=True)
        ic = compute_information_content(train_data, ttype, pe)
        ic_vecs[ont] = ic
        print(f"  {ont} ({ttype}): IC ready ({(ic > 0).sum().item()} terms with annotations)")
    except Exception as e:
        print(f"  {ont}: skipped -- {e}")
print("IC computation complete.")


In [ ]:
# ── Cell 7: Initialise Models ──────────────────────────────────────────────────
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.distmult import DistMult
from src.models.reward import RewardModule
from src.utils.logger import TrainingLogger

encoder      = CompGCN(train_data, cfg).to(DEVICE)
generator    = Generator(cfg).to(DEVICE)
discriminator = Discriminator(cfg).to(DEVICE)
distmult     = DistMult(hidden_dim=cfg['distmult']['hidden_dim']).to(DEVICE)
reward_module = RewardModule(cfg, distmult, discriminator).to(DEVICE)

total_params = sum(p.numel() for p in encoder.parameters()) + \
               sum(p.numel() for p in generator.parameters()) + \
               sum(p.numel() for p in discriminator.parameters())
print(f'Total trainable parameters: {total_params:,}')

logger = TrainingLogger(
    log_dir='/tmp/runs',
    csv_path=LOG_CSV,
)
print('Logger ready. Run `%load_ext tensorboard` then `%tensorboard --logdir /tmp/runs` to monitor.')

## Skip Phase 1 & Phase 2 Retraining (if checkpoints already exist on Drive)

If `compgcn_pretrained.pt` and `adversarial_checkpoint.pt` are already saved in
`CHECKPOINT_DIR`, run **Cell 7b** below to load them directly instead of running
Cell 8 (Phase 1) and Cell 9 (Phase 2), which retrain from scratch (~2-3 hours
combined).

After Cell 7b reports "Checkpoint verified," skip straight to **Cell 10 (Phase 3)**.


In [ ]:
# ── Cell 7b: Load Existing Phase 1+2 Checkpoint (skip retraining) ─────────────
# Run this INSTEAD of Cell 8 (Phase 1) and Cell 9 (Phase 2) below if those
# checkpoints already exist on Drive from a previous session.
import os, torch
from src.evaluation.metrics import evaluate_all

ckpt_path = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')
assert os.path.exists(ckpt_path), (
    f"Missing checkpoint: {ckpt_path}\n"
    f"Run Cell 8 (Phase 1) and Cell 9 (Phase 2) below instead."
)
ckpt = torch.load(ckpt_path, map_location=DEVICE)

encoder.load_state_dict(ckpt['encoder'])
generator.load_state_dict(ckpt['generator'])
discriminator.load_state_dict(ckpt['discriminator'])
print(f"Loaded Phase 2 checkpoint from {ckpt_path}")
print(f"  encoder params:       {sum(p.numel() for p in encoder.parameters()):,}")
print(f"  generator params:     {sum(p.numel() for p in generator.parameters()):,}")
print(f"  discriminator params: {sum(p.numel() for p in discriminator.parameters()):,}")

print("\nSanity check -- should land near Fmax 0.3484 (prior Phase-2 result):")
sanity = evaluate_all(
    encoder=encoder, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode="encoder", ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)
assert abs(sanity["fmax"] - 0.3484) < 0.01, (
    "Fmax doesn't match the prior Phase-2 run -- checkpoint may be wrong or stale."
)
print("Checkpoint verified. Skip to Cell 10 (Phase 3) below.")


In [ ]:
# ── Cell 8: Phase 1 — Pre-training ────────────────────────────────────────────
# SKIP THIS CELL if you already ran Cell 7b above to load an existing checkpoint.
# Trains CompGCN encoder with 4 losses (MSE, cosine, ranking, MMD).
# Generator/Discriminator are frozen during this phase.
# Expected: ~30-40 min on T4 for 50 epochs.

from src.training.pretrain import pretrain

encoder = pretrain(
    encoder=encoder,
    train_data=train_data,
    val_data=val_data,
    cfg=cfg,
    device=DEVICE,
    logger=logger,
)

# Save encoder checkpoint to Drive
import torch, os
ckpt_path = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained.pt')
torch.save(encoder.state_dict(), ckpt_path)
print(f'Saved pretrained encoder → {ckpt_path}')

In [ ]:
# ── Cell 8b: Plot Phase 1 Curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt, os

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
history = logger.history

def plot_metric(ax, key, label, color='steelblue'):
    if key in history:
        steps, vals = zip(*history[key])
        ax.plot(steps, vals, color=color, linewidth=1.5)
        ax.set_title(label); ax.set_xlabel('Step'); ax.grid(alpha=0.3)

for ax, (key, lbl, col) in zip(axes[:3], [
    ('loss/total', 'Total Pretrain Loss', 'steelblue'),
    ('val/cosine_similarity', 'Val Cosine Similarity', 'green'),
    ('embed/protein_norm_mean', 'Protein Emb Norm', 'orange'),
]):
    plot_metric(ax, key, lbl, col)

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'pretrain_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

In [ ]:
# ── Cell 9: Phase 2 — Adversarial Training ────────────────────────────────────
# SKIP THIS CELL if you already ran Cell 7b above to load an existing checkpoint.
# GAN training: Discriminator updated 2x per Generator update.
# Expected: ~60-90 min on T4 for 100 epochs.

from src.training.adversarial import train_adversarial

encoder, generator, discriminator = train_adversarial(
    encoder=encoder,
    generator=generator,
    discriminator=discriminator,
    distmult=distmult,
    train_data=train_data,
    val_data=val_data,
    ancestor_table=ancestor_table,
    cfg=cfg,
    device=DEVICE,
    logger=logger,
)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')
torch.save({
    'encoder': encoder.state_dict(),
    'generator': generator.state_dict(),
    'discriminator': discriminator.state_dict(),
}, ckpt_path)
print(f'Saved adversarial checkpoint → {ckpt_path}')

In [ ]:
# ── Cell 9b: Plot Phase 2 Curves ──────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (key, lbl, col) in zip(axes, [
    ('loss/disc_total',    'Discriminator Loss',       'crimson'),
    ('loss/gen',           'Generator Loss',            'steelblue'),
    ('disc/real_acc',      'D Accuracy (Real)',         'green'),
    ('disc/fake_acc',      'D Accuracy (Fake)',         'darkorange'),
    ('reward/distmult_mean', 'DistMult Score (mean)',   'purple'),
    ('val/fmax_bp',        'Val Fmax (BP)',             'black'),
]):
    plot_metric(ax, key, lbl, col)

# Reference line for ProtHGT baseline
if 'val/fmax_bp' in history:
    steps, _ = zip(*history['val/fmax_bp'])
    axes[5].axhline(0.7489, color='red', linestyle='--', label='ProtHGT baseline')
    axes[5].legend()

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'adversarial_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

In [ ]:
# ── Cell 10: Phase 3 — RL Fine-tuning ────────────────────────────────────────
# REINFORCE with GO hierarchy penalty. Semantic reward grows via curriculum.
# Best val Fmax checkpoint is auto-saved to Drive.
# Expected: ~30-45 min on T4 for 50 epochs.

from src.training.rl_trainer import train_rl

encoder, generator = train_rl(
    encoder=encoder,
    generator=generator,
    distmult=distmult,
    reward_module=reward_module,
    train_data=train_data,
    val_data=val_data,
    ancestor_table=ancestor_table,
    cfg=cfg,
    device=DEVICE,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger,
)

ckpt_path = os.path.join(CHECKPOINT_DIR, 'rl_final.pt')
torch.save({'encoder': encoder.state_dict(), 'generator': generator.state_dict()}, ckpt_path)
print(f'Saved final RL checkpoint → {ckpt_path}')

In [ ]:
# ── Cell 10b: Plot Phase 3 Curves ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.ravel()

for ax, (key, lbl, col) in zip(axes, [
    ('reward/total_mean',       'Mean Reward',              'steelblue'),
    ('reward/hierarchy_penalty','Hierarchy Penalty',         'crimson'),
    ('reward/semantic',         'Semantic Reward',           'green'),
    ('curriculum/w3',           'Semantic Weight w3(t)',     'orange'),
    ('grad/gen_norm',           'Generator Grad Norm',       'purple'),
    ('val/fmax_bp',             'Val Fmax (BP)',             'black'),
]):
    plot_metric(ax, key, lbl, col)

if 'val/fmax_bp' in history:
    axes[5].axhline(0.7489, color='red', linestyle='--', label='ProtHGT baseline')
    axes[5].legend()

plt.tight_layout()
img_path = os.path.join(CHECKPOINT_DIR, 'rl_curves.png')
plt.savefig(img_path, dpi=100)
plt.show()
print(f'Saved → {img_path}')

## Final Evaluation — Run These For The Poster

The three cells below produce the numbers and table for the poster:

- **Cell 11d** reloads each phase's own checkpoint independently (Phase 1,
  Phase 1+2, Phase 1+2+3) and scores each one the same way. This is the
  correct phase-vs-phase comparison — unlike Cell 11b further down, which
  compares *scoring modes* on whatever weights happen to be loaded, not phases.
- **Cell 11e** trains two shallow (no-graph) knowledge-graph-embedding
  baselines — plain ComplEx and plain DistMult — directly on the protein-GO
  triples, to show how much the CompGCN's graph structure actually buys you.
- **Cell 11f** merges both into the final table, with Fmax, Smin, AUPR, F1,
  MCC, Hit@10, and MRR for every row.


In [ ]:
# ── Cell 11d: True Phase Comparison (reloads each checkpoint independently) ───
# Fixes the bug in Cell 11b below, where "Baseline (encoder only)" and
# "+ Adversarial (encoder)" both scored the SAME in-memory weights (whatever
# was last trained) instead of actually comparing Phase 1 vs Phase 2 vs Phase 3.
import json, os, torch
from src.evaluation.metrics import evaluate_all, print_ablation_table
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator

for f in ['compgcn_pretrained.pt', 'adversarial_checkpoint.pt', 'rl_final.pt']:
    p = os.path.join(CHECKPOINT_DIR, f)
    print(f, '->', 'OK' if os.path.exists(p) else 'MISSING', p)

phase_results = {}

# --- Phase 1 only ---
enc1 = CompGCN(train_data, cfg).to(DEVICE)
enc1.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained.pt'), map_location=DEVICE))
print("\n=== Phase 1 (pretrain only) ===")
phase_results["Phase 1 (pretrain only)"] = {"bp": evaluate_all(
    encoder=enc1, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, baseline_fmax=0.7489, mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)}

# --- Phase 1 + 2 ---
ckpt2 = torch.load(os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt'), map_location=DEVICE)
enc2 = CompGCN(train_data, cfg).to(DEVICE); enc2.load_state_dict(ckpt2['encoder'])
gen2 = Generator(cfg).to(DEVICE); gen2.load_state_dict(ckpt2['generator'])
disc2 = Discriminator(cfg).to(DEVICE); disc2.load_state_dict(ckpt2['discriminator'])
print("\n=== Phase 1+2 (adversarial) ===")
phase_results["Phase 1+2 (adversarial)"] = {"bp": evaluate_all(
    encoder=enc2, generator=gen2, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, baseline_fmax=0.7489, mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]), discriminator=disc2,
)}

# --- Phase 1 + 2 + 3 ---
ckpt3 = torch.load(os.path.join(CHECKPOINT_DIR, 'rl_final.pt'), map_location=DEVICE)
enc3 = CompGCN(train_data, cfg).to(DEVICE); enc3.load_state_dict(ckpt3['encoder'])
gen3 = Generator(cfg).to(DEVICE); gen3.load_state_dict(ckpt3['generator'])
print("\n=== Phase 1+2+3 (full pipeline) ===")
phase_results["Phase 1+2+3 (full)"] = {"bp": evaluate_all(
    encoder=enc3, generator=gen3, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, baseline_fmax=0.7489, mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)}

print("\n" + "="*70 + "\n  TRUE PHASE COMPARISON (each row = its own checkpoint)\n" + "="*70)
print_ablation_table(phase_results)

with open(os.path.join(CHECKPOINT_DIR, "phase_comparison_results.json"), "w") as f:
    json.dump(phase_results, f, indent=2)
print(f"Saved -> {CHECKPOINT_DIR}/phase_comparison_results.json")


In [ ]:
# ── Cell 11e: Shallow KGE Baselines (ComplEx-only / DistMult-only, no graph) ──
# Plain embedding-table models trained directly on protein-GO triples, with no
# graph convolution at all. Isolates how much the CompGCN's message passing
# actually contributes over a standard shallow KGC baseline.
from src.models.distmult import DistMult, TrueDistMult   # DistMult class = ComplEx scoring
from src.models.shallow_kge import train_shallow_kge
from src.evaluation.metrics import evaluate_all

HIDDEN_DIM = cfg["encoder"]["output_dim"]

# --- Shallow ComplEx baseline (no graph) ---
complex_scorer = DistMult(hidden_dim=HIDDEN_DIM)
shallow_complex_encoder = train_shallow_kge(
    train_data, target_type, complex_scorer, HIDDEN_DIM,
    device=DEVICE, epochs=50,
)
print("\n=== Shallow ComplEx (no graph) ===")
results_shallow_complex = evaluate_all(
    encoder=shallow_complex_encoder, generator=None, distmult=complex_scorer,
    data=test_data, ancestor_table=ancestor_table, target_type=target_type,
    cfg=cfg, device=DEVICE, baseline_fmax=0.7489,
)

# --- Shallow DistMult baseline (no graph, symmetric) ---
distmult_scorer = TrueDistMult(hidden_dim=HIDDEN_DIM)
shallow_distmult_encoder = train_shallow_kge(
    train_data, target_type, distmult_scorer, HIDDEN_DIM,
    device=DEVICE, epochs=50,
)
print("\n=== Shallow DistMult (no graph) ===")
results_shallow_distmult = evaluate_all(
    encoder=shallow_distmult_encoder, generator=None, distmult=distmult_scorer,
    data=test_data, ancestor_table=ancestor_table, target_type=target_type,
    cfg=cfg, device=DEVICE, baseline_fmax=0.7489,
)


In [ ]:
# ── Cell 11f: Final Combined Results Table (for poster) ───────────────────────
# Requires Cell 11d (phase_results) and Cell 11e (results_shallow_*) to have run.
import json, os
from src.evaluation.metrics import print_ablation_table

final_results = {
    "DistMult only (shallow, no graph)": {"bp": results_shallow_distmult},
    "ComplEx only (shallow, no graph)":  {"bp": results_shallow_complex},
    **phase_results,
}

print("\n" + "="*70 + "\n  FINAL POSTER TABLE\n" + "="*70)
print_ablation_table(final_results)

with open(os.path.join(CHECKPOINT_DIR, "final_poster_results.json"), "w") as f:
    json.dump(final_results, f, indent=2)
print(f"Saved -> {CHECKPOINT_DIR}/final_poster_results.json")


In [ ]:
# Cell 11 (optional/legacy): Quick single-condition test evaluation.
# Superseded by Cell 11d (True Phase Comparison) above, which is the
# canonical source for poster numbers. Kept here for ad-hoc spot checks
# of whatever weights are currently loaded into `encoder`/`generator`.
import json, os
from src.evaluation.metrics import evaluate_all

PUBLISHED_FMAX = {
    "GO_term_P": 0.7489,   # ProtHGT-ESM2 BP (as reported in their paper)
    "GO_term_F": None,     # fill from ProtHGT paper once known
    "GO_term_C": None,
}

print("\n=== CONDITION 1: Encoder-only (ComplEx scorer) ===")
results_baseline = evaluate_all(
    encoder=encoder,
    generator=generator,
    distmult=distmult,
    data=test_data,
    ancestor_table=ancestor_table,
    target_type=target_type,
    cfg=cfg,
    device=DEVICE,
    baseline_fmax=PUBLISHED_FMAX.get(target_type),
    mode="encoder",
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
    discriminator=discriminator,
)

results_path = os.path.join(CHECKPOINT_DIR, "results_baseline.json")
with open(results_path, "w") as f:
    json.dump(results_baseline, f, indent=2)
print(f"Saved -> {results_path}")
logger.close()


In [ ]:
# Cell 11b (optional/legacy): Scoring-Mode Comparison (encoder vs critic vs
# ensemble), BP/MF/CC.
# IMPORTANT: this compares SCORING MODES on whatever weights are currently
# loaded into `encoder`/`generator`/`discriminator` -- it is NOT a Phase 1
# vs Phase 2 vs Phase 3 comparison. For that, use Cell 11d above, which
# reloads each phase's own checkpoint independently.
# Run AFTER Phase 2 (or Phase 3) completes (discriminator required for
# critic/ensemble modes). tune_ensemble_alpha() sweeps alpha on val data.
import json, os
from src.evaluation.metrics import evaluate_all, tune_ensemble_alpha, print_ablation_table
from src.data.loader import load_prothgt_splits

ONTOLOGY_TO_NODE = {"bp": "GO_term_P"}   # encoder trained on BP only

# -- Tune ensemble alpha on validation set (ComplEx vs critic weight) -----------
print("Tuning ensemble alpha on val set ...")
best_alpha = tune_ensemble_alpha(
    encoder=encoder,
    discriminator=discriminator,
    distmult=distmult,
    val_data=val_data,
    target_type=target_type,
    cfg=cfg,
    device=DEVICE,
)
print(f"Best alpha (ComplEx weight): {best_alpha:.2f}")
cfg["evaluation"]["ensemble_alpha"] = best_alpha

# -- Load splits for all three ontologies --------------------------------------
all_splits = {}
for ont in ["bp"]:   # encoder trained on BP only — skip MF/CC
    try:
        sp = load_prothgt_splits(cfg["data"]["drive_root"], ontology=ont, device="cpu")
        all_splits[ont] = (sp.train, sp.val, sp.test)
        print(f"  Loaded {ont}")
    except Exception as e:
        print(f"  {ont} skipped: {e}")

# -- Build ancestor tables and IC vecs for each ontology -----------------------
from src.data.go_hierarchy import build_ancestor_table, build_propagation_edges
from src.evaluation.smin import compute_information_content

anc_tables, ic_all = {}, {}
for ont, ttype in ONTOLOGY_TO_NODE.items():
    if ont not in all_splits:
        continue
    train_d, _, _ = all_splits[ont]
    cache_p = f"/tmp/ancestor_table_{ttype}.pkl"
    anc_tables[ont] = build_ancestor_table(train_d, target_type=ttype, cache=True,
                                           cache_path=cache_p)
    try:
        pe = build_propagation_edges(train_d, target_type=ttype, cache=True)
        ic_all[ont] = compute_information_content(train_d, ttype, pe)
    except Exception as e:
        print(f"  IC {ont}: {e}")

# -- Run 4 ablation conditions -------------------------------------------------
conditions = [
    # NOTE: removed the duplicate "Baseline (encoder only)" / "+ Adversarial
    # (encoder)" rows -- both used mode="encoder" on the SAME currently-loaded
    # weights, so they were always identical. This now lists each scoring mode once.
    ("Scoring mode: encoder (ComplEx)", "encoder",  None),
    ("Scoring mode: critic",            "critic",   None),
    ("Scoring mode: ensemble",          "ensemble", best_alpha),
]

ablation_results = {}
for cond_name, mode, alpha in conditions:
    ablation_results[cond_name] = {}
    cfg["evaluation"]["mode"] = mode
    if alpha is not None:
        cfg["evaluation"]["ensemble_alpha"] = alpha

    for ont, ttype in ONTOLOGY_TO_NODE.items():
        if ont not in all_splits:
            continue
        _, _, test_d = all_splits[ont]
        print(f"\n{cond_name} | {ont.upper()} ...")
        ablation_results[cond_name][ont] = evaluate_all(
            encoder=encoder, generator=generator, distmult=distmult,
            data=test_d, ancestor_table=anc_tables[ont],
            target_type=ttype, cfg=cfg, device=DEVICE,
            mode=mode,
            ensemble_alpha=alpha if alpha is not None else cfg["evaluation"]["ensemble_alpha"],
            ic_vec=ic_all.get(ont),
            discriminator=discriminator,
        )

# -- Print and save ------------------------------------------------------------
print("\n" + "=" * 70)
print("  ABLATION TABLE")
print("=" * 70)
print_ablation_table(ablation_results)

with open(os.path.join(CHECKPOINT_DIR, "ablation_results.json"), "w") as f:
    json.dump(ablation_results, f, indent=2)
print(f"\nSaved -> {CHECKPOINT_DIR}/ablation_results.json")


In [ ]:
# Cell 11c: ProtHGT Re-run Through Our Metric Harness
# Run ProtHGT inference on the same .pt splits, then evaluate through identical
# propagation/threshold code so the comparison is apples-to-apples.
#
# Step 1 (run once): Clone ProtHGT and install deps
# !git clone https://github.com/HanwenXuTHU/ProtHGT /content/ProtHGT
# %cd /content/ProtHGT && pip install -r requirements.txt -q && %cd /content/Prot_B_poster
#
# Step 2: Run their inference script and save scores as [N_test_p, N_go] tensors
# !python /content/ProtHGT/eval.py \
#     --data_root /content/drive/MyDrive/Poster\ code/prothgt/knowledge_graphs/ \
#     --output_dir /content/drive/MyDrive/Poster\ code/checkpoints/ \
#     --ontology bp
#
# Step 3 (this cell): Load saved scores and evaluate
import os, json, torch
from src.data.go_hierarchy import build_propagation_edges
from src.data.graph_builder import build_annotation_matrix
import numpy as np

PUBLISHED = {"bp": 0.7489, "mf": None, "cc": None}

prothgt_row = {}
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    score_file = os.path.join(CHECKPOINT_DIR, f"prothgt_scores_{ont}.pt")
    if not os.path.exists(score_file):
        print(f"  {ont}: {score_file} not found -- skipping")
        continue
    if ont not in all_splits:
        print(f"  {ont}: split not loaded -- skipping")
        continue

    prothgt_scores = torch.load(score_file)   # [N_test_p, N_go]
    _, _, test_d = all_splits[ont]

    pe  = build_propagation_edges(test_d, target_type=ttype, cache=True)
    row, col, n_p, n_go = build_annotation_matrix(test_d, ttype)
    true_mat = torch.zeros(n_p, n_go, dtype=torch.float32)
    true_mat[row.cpu(), col.cpu()] = 1.0

    # Propagate predictions and labels up the GO DAG
    for child_i, parent_i in pe:
        prothgt_scores[:, parent_i] = torch.max(prothgt_scores[:, parent_i],
                                                 prothgt_scores[:, child_i])
        true_mat[:, parent_i] = torch.max(true_mat[:, parent_i],
                                           true_mat[:, child_i])
    true_mat = (true_mat > 0.5).float()

    has_annot = true_mat.any(dim=1)
    sm = prothgt_scores[has_annot]
    tm = true_mat[has_annot]
    ct_sum = tm.sum(1).clamp(min=1e-8)

    s_min_v, s_max_v = sm.min().item(), sm.max().item()
    t_steps = cfg.get("evaluation", {}).get("threshold_steps", 100)
    best_f1, best_t = 0.0, s_min_v
    for i in range(t_steps + 1):
        t    = s_min_v + i * (s_max_v - s_min_v) / t_steps
        pred = (sm >= t).float()
        tp   = (pred * tm).sum(1)
        prec = (tp / pred.sum(1).clamp(1e-8)).mean().item()
        rec  = (tp / ct_sum).mean().item()
        if prec + rec > 0:
            f1 = 2 * prec * rec / (prec + rec)
            if f1 > best_f1:
                best_f1, best_t = f1, t

    prothgt_row[ont] = {"fmax": best_f1, "best_threshold": best_t}
    print(f"  ProtHGT-ESM2 (our harness) | {ont.upper()} | Fmax={best_f1:.4f}")

# Add to ablation table as reference rows
ablation_results["ProtHGT-ESM2 (our harness)"] = prothgt_row
ablation_results["ProtHGT-ESM2 (as reported)"] = {
    ont: {"fmax": v} for ont, v in PUBLISHED.items() if v is not None
}

from src.evaluation.metrics import print_ablation_table
print("\nFull ablation table with ProtHGT reference rows:")
print_ablation_table(ablation_results)

with open(os.path.join(CHECKPOINT_DIR, "ablation_results_final.json"), "w") as f:
    json.dump(ablation_results, f, indent=2)
print(f"Saved -> {CHECKPOINT_DIR}/ablation_results_final.json")


In [ ]:
# ── Optional: TensorBoard ──────────────────────────────────────────────────────
# Run this cell at any time to open TensorBoard and see live training curves.
%load_ext tensorboard
%tensorboard --logdir /tmp/runs/

In [ ]:
# ── Optional: Resume from Checkpoint ──────────────────────────────────────────
# If Colab disconnects mid-training, use this cell to reload from the last checkpoint.

RESUME_PHASE = 'adversarial'   # 'pretrain' | 'adversarial' | 'rl'
RESUME_PATH = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')  # adjust as needed

ckpt = torch.load(RESUME_PATH, map_location=DEVICE)

if RESUME_PHASE == 'pretrain':
    encoder.load_state_dict(ckpt)
    print('Loaded pretrained encoder.')
elif RESUME_PHASE in ('adversarial', 'rl'):
    encoder.load_state_dict(ckpt['encoder'])
    generator.load_state_dict(ckpt['generator'])
    if 'discriminator' in ckpt:
        discriminator.load_state_dict(ckpt['discriminator'])
    print(f'Loaded {RESUME_PHASE} checkpoint.')